# StudyMate — 학습 주제 → 카드 + 퀴즈

데모 데이 Day 1. 강의 #15.0 ~ #15.5에서 본 LangGraph 패턴(`StateGraph`, `Send` 디스패치)을 그대로 응용해서,
학습 주제 한 줄을 받으면 **핵심 개념 → 플래시카드 → 객관식 퀴즈**까지 한 번에 만들어주는 에이전트.

## Step 1 — 설계

### 이름
**StudyMate**

### 목적
공부할 때 매번 직접 정리 → 카드 만들기 → 문제 풀어보기 사이클을 도는데,
정리/카드 만드는 단계가 너무 시간을 잡아먹는다. LLM이 한 방에 만들어주면
학습자는 **회상(recall)** 과 **검증(quiz)** 단계에만 집중하면 된다.

### 핵심 기능 (3가지)
1. **개념 추출** — 주제 → 학습 포인트 5개 (LLM 1회 호출)
2. **플래시카드 생성** — 각 개념별 Q&A 카드, `Send`로 5장 동시 생성
3. **이해도 퀴즈** — 5개 개념을 묶어 객관식 4문제 + 정답 인덱스

### 그래프 구조

```
          START
            │
            ▼
      analyze_topic        — 핵심 개념 5개 추출
            │
            ▼
    (dispatch_cards) ── Send ──┐
                               ▼
                      make_flashcard × 5  (병렬)
                               │
                               ▼
                          make_quiz       — 객관식 4문제
                               │
                               ▼
                              END
```

## Step 2 — 기초 구축

강의에서 본 패턴 그대로:
- `TypedDict` State + `Annotated[..., operator.add]` 누적 필드
- `StateGraph(State)` 빌더
- `Send`로 fan-out 병렬 처리 (썸네일 5장 만들던 것과 동일)

In [ ]:
# 환경 변수 로드 (.env에 OPENAI_API_KEY)
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langgraph.graph import START, END, StateGraph
from langgraph.types import Send
from langchain.chat_models import init_chat_model
from typing import TypedDict
from typing_extensions import Annotated
import operator
import json

llm = init_chat_model("openai:gpt-4o-mini")

### State

`flashcards`만 누적 필드(`operator.add`). 5개 노드가 병렬로 카드 1장씩 던져도
리스트로 합쳐진다. 나머지는 단발성.

In [ ]:
class State(TypedDict):
    topic: str
    concepts: list[str]
    flashcards: Annotated[list[dict], operator.add]
    quiz: list[dict]

### JSON 파싱 유틸

gpt-4o-mini가 가끔 \`\`\`json ... \`\`\` 코드블록을 두르고 답해서, 한 번 벗겨준다.

In [ ]:
def parse_json(text: str):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").strip()
        if text.startswith("json"):
            text = text[4:].strip()
    return json.loads(text)

### Node 1 — `analyze_topic`
주제를 받아 핵심 개념 5개를 명사구로 뽑는다.

In [ ]:
def analyze_topic(state: State):
    prompt = f"""
    학습 주제: {state["topic"]}

    위 주제를 처음 공부하는 학습자가 반드시 알아야 할 핵심 개념을 정확히 5개 뽑아줘.
    각 항목은 짧은 한국어 명사구.
    출력은 JSON 배열만, 예: ["개념1", "개념2", ...]
    """
    response = llm.invoke(prompt)
    concepts = parse_json(response.content)
    return {"concepts": concepts}

### Dispatch — `dispatch_cards`
강의의 `dispatch_artists`처럼 conditional edge에서 `Send` 리스트를 던져 5개 노드를 병렬 실행.

In [ ]:
def dispatch_cards(state: State):
    return [
        Send("make_flashcard", {"concept": c, "topic": state["topic"]})
        for c in state["concepts"]
    ]

### Node 2 — `make_flashcard`
개념 1개를 받아 Q&A 카드 한 장 생성. 노드 인자가 `state` 전체가 아니라 `Send`로 던진 dict인 점이 포인트.

In [ ]:
def make_flashcard(args):
    prompt = f"""
    주제: {args["topic"]}
    개념: {args["concept"]}

    위 개념을 학습용 플래시카드로 만들어줘. 형식은 JSON:
    {{"question": "...", "answer": "..."}}
    answer는 2~3문장 한국어로 간결하게.
    """
    response = llm.invoke(prompt)
    card = parse_json(response.content)
    return {"flashcards": [card]}

### Node 3 — `make_quiz`
5개 개념을 다 합쳐 객관식 4문제. fan-in 자리.

In [ ]:
def make_quiz(state: State):
    concepts = "\n".join(f"- {c}" for c in state["concepts"])
    prompt = f"""
    주제: {state["topic"]}
    핵심 개념:
    {concepts}

    위 개념들의 이해도를 점검할 객관식 4문제를 만들어줘.
    각 문제는 4지선다, 정답 인덱스(0~3) 포함. 출력은 JSON 배열:
    [{{"q": "...", "options": ["a","b","c","d"], "answer_index": 0}}, ...]
    """
    response = llm.invoke(prompt)
    quiz = parse_json(response.content)
    return {"quiz": quiz}

### 그래프 연결

In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node("analyze_topic", analyze_topic)
graph_builder.add_node("make_flashcard", make_flashcard)
graph_builder.add_node("make_quiz", make_quiz)

graph_builder.add_edge(START, "analyze_topic")
graph_builder.add_conditional_edges(
    "analyze_topic", dispatch_cards, ["make_flashcard"]
)
graph_builder.add_edge("make_flashcard", "make_quiz")
graph_builder.add_edge("make_quiz", END)

graph = graph_builder.compile(name="study_mate")

### 그래프 시각화 확인

In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

## 실행 예시

주제 하나 던져서 카드 5장 + 퀴즈 4문제가 나오는지 확인.

In [ ]:
result = graph.invoke({"topic": "FastAPI 의존성 주입(Dependency Injection)"})

print("=== 핵심 개념 ===")
for c in result["concepts"]:
    print(" -", c)

print("\n=== 플래시카드 ===")
for i, card in enumerate(result["flashcards"], 1):
    print(f"[{i}] Q. {card['question']}")
    print(f"    A. {card['answer']}\n")

print("=== 퀴즈 ===")
for i, q in enumerate(result["quiz"], 1):
    print(f"{i}. {q['q']}")
    for j, opt in enumerate(q["options"]):
        mark = "✓" if j == q["answer_index"] else " "
        print(f"   {mark} {j}) {opt}")

## Day 2 이후 계획

- 카드 보고 "어려움/쉬움" 표시 → 어려운 개념만 카드 보강 루프
- 퀴즈 채점 후 틀린 문제 개념만 다시 학습
- `interrupt()`로 인터랙티브 학습 세션 (강의 #15.5 패턴)
- 다국어 지원 (영어 단어 학습용)